# 🏊🚴🏃 IRONMAN 70.3 — Sistema de Análisis Diario
### Equipo Incubadora | Analista: Simón (Datos) | v2.0 con Filtros Expertos

> **Objetivo:** Dashboard completo con estado actual, progreso por disciplina, plan adaptativo basado en HRV y recomendaciones del equipo.

**Datos:** Apple Watch Auto Export → iCloud → sincronizado automáticamente cada mañana  
**Filtros:** `REGLAS_FILTRADO_IRONMAN.md` — solo sesiones con impacto real en preparación Ironman 70.3


In [ ]:
import pandas as pd, json, os, glob, warnings
from datetime import datetime, timedelta, date
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np

warnings.filterwarnings('ignore')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT_DIR  = "/Users/Lobsang/Library/Mobile Documents/com~apple~CloudDocs/Personal Knowledge Assitant/Incubadora/Ironman_70_3_Project"
WORKOUTS_DIR = os.path.join(PROJECT_DIR, "Workouts Apple Watch")
HEALTH_DIR   = os.path.join(PROJECT_DIR, "health data apple watch")
REPORTES_DIR = os.path.join(PROJECT_DIR, "reportes")
os.makedirs(REPORTES_DIR, exist_ok=True)

# ── REGLAS DE FILTRADO IRONMAN ─────────────────────────────────────
FILTROS_IRONMAN = {
    'Pool Swim':                        {'min_km': 0,   'min_min': 0,  'disciplina': '🏊 Natacion'},
    'Open Water Swim':                  {'min_km': 0,   'min_min': 0,  'disciplina': '🏊 Natacion'},
    'Outdoor Cycling':                  {'min_km': 8,   'min_min': 0,  'disciplina': '🚴 Ciclismo'},
    'Indoor Cycling':                   {'min_km': 0,   'min_min': 20, 'disciplina': '🚴 Ciclismo'},
    'Outdoor Run':                      {'min_km': 3,   'min_min': 0,  'disciplina': '🏃 Running'},
    'Indoor Run':                       {'min_km': 3,   'min_min': 0,  'disciplina': '🏃 Running'},
    'Outdoor Walk':                     {'min_km': 5,   'min_min': 0,  'disciplina': '🚶 Fuerza Base'},
    'Hiking':                           {'min_km': 5,   'min_min': 0,  'disciplina': '🚶 Fuerza Base'},
    'Traditional Strength Training':    {'min_km': 0,   'min_min': 0,  'disciplina': '�� Fuerza'},
    'Core Training':                    {'min_km': 0,   'min_min': 0,  'disciplina': '💪 Fuerza'},
    'High Intensity Interval Training': {'min_km': 0,   'min_min': 0,  'disciplina': '⚡ HIIT'},
}

COLORES = {
    '🏊 Natacion':'#3498DB', '🚴 Ciclismo':'#E67E22', '🏃 Running':'#2ECC71',
    '💪 Fuerza':'#E74C3C',   '🚶 Fuerza Base':'#9B59B6', '⚡ HIIT':'#F1C40F',
}

def safe_qty(field):
    if isinstance(field, list) and field:
        vals = [x.get('qty', 0) for x in field if isinstance(x, dict) and 'qty' in x]
        return round(sum(vals)/len(vals), 2) if vals else None
    if isinstance(field, dict): return field.get('qty')
    return None

print("✅ Setup OK — FILTROS_IRONMAN cargados:", len(FILTROS_IRONMAN), "tipos de actividad")


## 📊 Simón — Carga y Filtrado de Datos
> Carga JSONs Apple Watch, aplica filtros y construye DataFrames para el equipo.

In [ ]:
# ── HEALTH DATA ───────────────────────────────────────────────────
METRICAS = ['heart_rate_variability','resting_heart_rate','vo2_max',
            'blood_oxygen_saturation','step_count','active_energy','cardio_recovery']
h_records = []
for fpath in sorted(glob.glob(os.path.join(HEALTH_DIR, '*.json'))):
    try:
        raw = json.load(open(fpath, encoding='utf-8'))
        row = {}
        for m in raw.get('data',{}).get('metrics',[]):
            nm = m.get('name','')
            if nm in METRICAS:
                pts = m.get('data',[])
                vals = [d.get('qty', d.get('Avg')) for d in pts if d]
                vals = [v for v in vals if v is not None]
                row[nm] = round(sum(vals)/len(vals),2) if vals else None
                if 'fecha' not in row and pts:
                    row['fecha'] = pd.to_datetime(pts[0].get('date',''), utc=True, errors='coerce')
        if row: h_records.append(row)
    except: pass

dh = pd.DataFrame(h_records)
dh['fecha'] = dh['fecha'].dt.tz_localize(None)
dh = dh.sort_values('fecha').reset_index(drop=True)
print(f"✅ Health data: {len(dh)} dias | {dh['fecha'].min().date()} - {dh['fecha'].max().date()}")
print(f"   HRV: {dh['heart_rate_variability'].notna().sum()}d | RHR: {dh['resting_heart_rate'].notna().sum()}d | VO2: {dh['vo2_max'].notna().sum()}d")

# ── WORKOUTS CON FILTRADO IRONMAN ─────────────────────────────────
all_recs, filtered_out = [], []
for fpath in sorted(glob.glob(os.path.join(WORKOUTS_DIR, '*.json'))):
    try:
        raw = json.load(open(fpath, encoding='utf-8'))
        for w in raw.get('data',{}).get('workouts',[]):
            name = w.get('name','')
            if name not in FILTROS_IRONMAN: continue
            dur = w.get('duration')
            dur_min = None
            if isinstance(dur, dict):
                q,u = dur.get('qty'), dur.get('units','')
                if q: dur_min = q/60 if 'sec' in u.lower() else q
            dist = w.get('distance')
            dist_km = None
            if isinstance(dist, dict):
                q,u = dist.get('qty'), dist.get('units','').lower()
                if q: dist_km = q*1.60934 if 'mi' in u else q
            elif isinstance(dist, list) and dist:
                q,u = dist[0].get('qty'), dist[0].get('units','').lower()
                if q: dist_km = q*1.60934 if 'mi' in u else q
            regla = FILTROS_IRONMAN[name]
            pasa = (dist_km or 0)>=regla['min_km'] and (dur_min or 0)>=regla['min_min']
            rec = {
                'fecha': pd.to_datetime(w.get('start',''), utc=True, errors='coerce'),
                'nombre': name, 'disciplina': regla['disciplina'],
                'duracion_min': round(dur_min,1) if dur_min else None,
                'distancia_km': round(dist_km,2) if dist_km else None,
                'hr_avg': safe_qty(w.get('avgHeartRate')),
                'hr_max': safe_qty(w.get('maxHeartRate')),
                'calorias': safe_qty(w.get('activeEnergyBurned')),
            }
            (all_recs if pasa else filtered_out).append(rec)
    except: pass

df = pd.DataFrame(all_recs).dropna(subset=['fecha'])
df['fecha'] = df['fecha'].dt.tz_localize(None)
df = df.sort_values('fecha').reset_index(drop=True)
df['semana'] = df['fecha'].dt.to_period('W').dt.start_time

print(f"\n✅ Sesiones relevantes Ironman: {len(df)}")
print(f"⬛ Sesiones descartadas:         {len(filtered_out)}")
print(f"\nResumen por disciplina:")
print(df.groupby('disciplina').agg(
    sesiones=('nombre','count'), km_total=('distancia_km','sum'),
    km_max=('distancia_km','max'), hr_prom=('hr_avg','mean')
).round(1).to_string())


## 📍 Estado Actual — Alejandro (Médico)
> Evaluación clínica del atleta: métricas fisiológicas, semáforo HRV y brechas vs Ironman 70.3.

In [ ]:
# ── ESTADO ACTUAL ─────────────────────────────────────────────────
hoy     = df['fecha'].max()
hace_7  = hoy - timedelta(days=7)
hace_14 = hoy - timedelta(days=14)
hace_28 = hoy - timedelta(days=28)

hrv_global  = dh['heart_rate_variability'].dropna().mean()
hrv_7d      = dh[dh['fecha']>=hace_7]['heart_rate_variability'].dropna().mean()
rhr_7d      = dh[dh['fecha']>=hace_7]['resting_heart_rate'].dropna().mean()
vo2_actual  = dh['vo2_max'].dropna().iloc[-1]
vo2_inicio  = dh['vo2_max'].dropna().iloc[0]

km_nado_max = df[df['disciplina']=='🏊 Natacion']['distancia_km'].max() or 0
km_bici_max = df[df['disciplina']=='🚴 Ciclismo']['distancia_km'].max() or 0
km_run_max  = df[df['disciplina']=='🏃 Running']['distancia_km'].max() or 0
ses_7d      = len(df[df['fecha']>=hace_7])
ses_14d     = len(df[df['fecha']>=hace_14])

# Semaforo HRV
if hrv_7d >= hrv_global*0.95:   semaforo,accion = "🟢 VERDE",   "Entrenar segun plan completo"
elif hrv_7d >= hrv_global*0.85: semaforo,accion = "🟡 AMARILLO","Reducir intensidad 20%, sin HIIT"
else:                             semaforo,accion = "🔴 ROJO",    "Solo descanso activo hoy"

print("="*62)
print(f"ESTADO ACTUAL DE LOBSANG — {hoy.strftime('%d %b %Y')}")
print("="*62)
print(f"\n  METRICAS FISIOLOGICAS:")
print(f"    VO2 Max:  {vo2_actual:.1f} ml/kg/min  (inicio: {vo2_inicio:.1f}  Delta: +{vo2_actual-vo2_inicio:.1f}) ✅")
print(f"    HRV 7d:   {hrv_7d:.1f} ms  (media global: {hrv_global:.1f} ms)  → {semaforo}")
print(f"    Accion:   {accion}")
print(f"    RHR 7d:   {rhr_7d:.1f} BPM  {'⚠️ Atención >65' if rhr_7d>65 else '✅ OK'}")
print(f"\n  SESIONES RECIENTES:")
print(f"    Ultimos 7 dias:   {ses_7d} sesiones")
print(f"    Ultimos 14 dias:  {ses_14d} sesiones")
print(f"\n  BRECHAS vs IRONMAN 70.3 (distancia maxima lograda vs meta carrera):")
for disc,actual,meta in [('Natacion',km_nado_max,1.9),('Ciclismo',km_bici_max,90),('Running',km_run_max,21.1)]:
    pct = actual/meta*100 if meta>0 else 0
    bar = '█'*int(pct/5) + '░'*(20-int(pct/5))
    print(f"    {disc:<10} [{bar}] {pct:.0f}%  ({actual:.1f} / {meta} km)")
print(f"    VO2 Max    [{'█'*int(vo2_actual/45*20/1)+'░'*(20-int(vo2_actual/45*20/1))}] {vo2_actual/45*100:.0f}%  ({vo2_actual:.1f} / 45 ml/kg/min)")
print(f"\n  ALERTAS MEDICAS ACTIVAS:")
print(f"    ⚠️  Isquiotibial izquierdo: recuperacion activa — protocolo CACO 4:1")
print(f"    ⚠️  Hombro: tension en ciclismo — revisar postura Azzurri")
print(f"    {'⚠️  HRV tendencia baja — posible fatiga acumulada' if hrv_7d < hrv_global*0.92 else '✅ HRV dentro del rango aceptable'}")


## 📈 Dashboard — Simón + Alejandro
> Visualización completa de progresión, fitness y carga de entrenamiento.

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.35)

# Panel 1: Km acumulados (full width)
ax1 = fig.add_subplot(gs[0,:])
for disc in ['🏊 Natacion','🚴 Ciclismo','🏃 Running','🚶 Fuerza Base']:
    sub = df[df['disciplina']==disc].dropna(subset=['distancia_km']).copy()
    if not len(sub): continue
    sub['km_acum'] = sub['distancia_km'].cumsum()
    ax1.plot(sub['fecha'], sub['km_acum'], marker='o', markersize=4,
             linewidth=2, color=COLORES[disc], label=disc)
ax1.set_title('Km Acumulados por Disciplina — Solo sesiones con impacto Ironman', fontweight='bold')
ax1.set_ylabel('km acumulados'); ax1.legend(loc='upper left',fontsize=9); ax1.grid(axis='y',alpha=0.3)

# Panel 2: VO2 Max
ax2 = fig.add_subplot(gs[1,0])
v = dh[['fecha','vo2_max']].dropna()
ax2.plot(v['fecha'], v['vo2_max'], color='#27AE60', linewidth=2, marker='s', markersize=4)
ax2.axhline(40, color='blue', ls='--', lw=1, label='Bueno (40)')
ax2.axhline(45, color='green', ls='--', lw=1, label='Excelente (45)')
ax2.fill_between(v['fecha'],40,45, alpha=0.07, color='green')
ax2.set_title('VO2 Max — Capacidad Aerobica', fontweight='bold')
ax2.set_ylabel('ml/kg/min'); ax2.legend(fontsize=8); ax2.grid(axis='y',alpha=0.3)

# Panel 3: HRV semaforo
ax3 = fig.add_subplot(gs[1,1])
h = dh[['fecha','heart_rate_variability']].dropna()
h_roll = h['heart_rate_variability'].rolling(7,min_periods=1).mean()
ax3.fill_between(h['fecha'], hrv_global*0.85, hrv_global*1.15, alpha=0.08, color='green')
ax3.fill_between(h['fecha'], hrv_global*0.75, hrv_global*0.85, alpha=0.08, color='yellow')
ax3.plot(h['fecha'], h['heart_rate_variability'], color='#8E44AD', lw=1, marker='o', ms=2, alpha=0.5)
ax3.plot(h['fecha'], h_roll, color='#6C3483', lw=2.5, label='Media 7d')
ax3.axhline(hrv_global, color='gray', ls='--', lw=1.2, label=f'Media global ({hrv_global:.0f}ms)')
ax3.set_title('HRV — Semaforo de Recuperacion', fontweight='bold')
ax3.set_ylabel('ms'); ax3.legend(fontsize=7,loc='lower right'); ax3.grid(axis='y',alpha=0.3)

# Panel 4: Carga semanal
ax4 = fig.add_subplot(gs[2,0])
discs_ord = ['🏊 Natacion','🚴 Ciclismo','🏃 Running','💪 Fuerza','🚶 Fuerza Base','⚡ HIIT']
carga = df.groupby(['semana','disciplina'])['duracion_min'].sum().unstack(fill_value=0)
cols_pres = [c for c in discs_ord if c in carga.columns]
bottom = np.zeros(len(carga))
for col in cols_pres:
    vals = carga[col].values.astype(float)
    ax4.bar(range(len(carga)), vals, bottom=bottom, label=col, color=COLORES.get(col,'gray'), width=0.8)
    bottom += vals
step = max(1, len(carga)//6)
ax4.set_xticks(range(0,len(carga),step))
ax4.set_xticklabels([str(d.date()) for d in carga.index[::step]], rotation=45, fontsize=7)
ax4.set_title('Carga Semanal por Disciplina (minutos)', fontweight='bold')
ax4.set_ylabel('Minutos'); ax4.legend(fontsize=7,loc='upper left'); ax4.grid(axis='y',alpha=0.3)

# Panel 5: RHR
ax5 = fig.add_subplot(gs[2,1])
r = dh[['fecha','resting_heart_rate']].dropna()
ax5.plot(r['fecha'], r['resting_heart_rate'], color='#E74C3C', lw=1, marker='o', ms=2, alpha=0.5)
ax5.plot(r['fecha'], r['resting_heart_rate'].rolling(7,min_periods=1).mean(), color='#C0392B', lw=2.5, label='Media 7d')
ax5.axhline(60, color='green', ls='--', lw=1, label='Meta (<60 BPM)')
ax5.axhline(65, color='orange', ls='--', lw=1, label='Atencion (65)')
ax5.set_title('Resting Heart Rate — Indicador de Forma', fontweight='bold')
ax5.set_ylabel('BPM'); ax5.legend(fontsize=8); ax5.grid(axis='y',alpha=0.3)

fig.suptitle(f'Dashboard Ironman 70.3 — Lobsang | {hoy.strftime("%d %b %Y")} | {len(df)} sesiones relevantes', fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(REPORTES_DIR,'dashboard_ironman.png'), dpi=130, bbox_inches='tight')
plt.show()
print("✅ Dashboard guardado → reportes/dashboard_ironman.png")


## 🏊🚴🏃 Disciplinas — Héctor + Nairo + Catherine
> Análisis individual de natación, ciclismo y running.

In [ ]:
# ── ANALISIS POR DISCIPLINA ───────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
fig.suptitle('Progresion por Disciplina Ironman 70.3', fontsize=13, fontweight='bold')
plt.subplots_adjust(hspace=0.45, wspace=0.3)

configs = [
    ('🏊 Natacion',  1.9,  '#3498DB', 'Hector'),
    ('🚴 Ciclismo',  90,   '#E67E22', 'Nairo'),
    ('🏃 Running',   21.1, '#2ECC71', 'Catherine'),
]

for i, (disc, meta_km, color, coach) in enumerate(configs):
    sub = df[df['disciplina']==disc].copy()
    ax_l, ax_r = axes[i][0], axes[i][1]

    # Distancia por sesion
    if len(sub)>0 and sub['distancia_km'].notna().sum()>0:
        ax_l.bar(range(len(sub)), sub['distancia_km'].fillna(0), color=color, alpha=0.7, width=0.8)
        ax_l.axhline(meta_km, color='red', ls='--', lw=1.5, label=f'Meta: {meta_km} km')
        roll = sub['distancia_km'].rolling(3, min_periods=1).mean()
        ax_l.plot(range(len(sub)), roll, color='darkred', lw=2, label='Tendencia 3ses')
        ax_l.set_title(f'{disc} | Coach: {coach} — Distancia/sesion', fontweight='bold', fontsize=9)
        ax_l.set_ylabel('km'); ax_l.legend(fontsize=7)
    else:
        ax_l.text(0.5,0.5,f'Sin datos\n{disc}',ha='center',va='center',transform=ax_l.transAxes,color='gray')
    ax_l.grid(axis='y',alpha=0.3)

    # FC promedio
    sub_hr = sub.dropna(subset=['hr_avg'])
    if len(sub_hr)>0:
        scatter_c = [color if h<160 else '#E74C3C' for h in sub_hr['hr_avg']]
        ax_r.scatter(range(len(sub_hr)), sub_hr['hr_avg'], c=scatter_c, s=50, alpha=0.8, zorder=3)
        ax_r.plot(range(len(sub_hr)), sub_hr['hr_avg'].rolling(3,min_periods=1).mean(),
                  color='gray', lw=1.5, ls='--', label='Tendencia')
        ax_r.axhline(155, color='orange', ls='--', lw=1, label='Umbral Z4 (155)')
        ax_r.axhline(143, color='green', ls='--', lw=1, label='Zona 2 (143)')
        ax_r.set_title(f'FC promedio por sesion', fontweight='bold', fontsize=9)
        ax_r.set_ylabel('BPM'); ax_r.legend(fontsize=7)
    else:
        ax_r.text(0.5,0.5,'Sin datos FC',ha='center',va='center',transform=ax_r.transAxes,color='gray')
    ax_r.grid(axis='y',alpha=0.3)

plt.savefig(os.path.join(REPORTES_DIR,'disciplinas_ironman.png'), dpi=120, bbox_inches='tight')
plt.show()
print("✅ Disciplinas guardado → reportes/disciplinas_ironman.png")


## 🧠🥗 Valentina + Andrea — Consistencia y Nutrición

In [ ]:
# ── VALENTINA: CONSISTENCIA ───────────────────────────────────────
ses_semana = df.groupby('semana').size()
semanas_excelentes = (ses_semana>=4).sum()
semanas_ok         = ((ses_semana>=2)&(ses_semana<4)).sum()
semanas_pobres     = (ses_semana<2).sum()
adherencia         = semanas_excelentes/len(ses_semana)*100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Valentina — Consistencia | Andrea — Distribucion de Esfuerzo', fontsize=12, fontweight='bold')

# Sesiones por semana
ax1 = axes[0]
colors_bar = ['#27AE60' if s>=4 else '#F39C12' if s>=2 else '#E74C3C' for s in ses_semana]
ax1.bar(range(len(ses_semana)), ses_semana.values, color=colors_bar, width=0.8)
ax1.axhline(4, color='green', ls='--', lw=1.5, label='Meta: 4+ ses/sem')
ax1.axhline(2, color='orange', ls='--', lw=1.5, label='Minimo: 2 ses/sem')
patches = [mpatches.Patch(color='#27AE60',label='Excelente >=4'),
           mpatches.Patch(color='#F39C12',label='OK 2-3'),
           mpatches.Patch(color='#E74C3C',label='Pobre <2')]
ax1.legend(handles=patches, fontsize=8)
step = max(1, len(ses_semana)//6)
ax1.set_xticks(range(0,len(ses_semana),step))
ax1.set_xticklabels([str(d.date()) for d in ses_semana.index[::step]], rotation=45, fontsize=7)
ax1.set_title(f'Sesiones por Semana | Adherencia excelente: {adherencia:.0f}%', fontweight='bold')
ax1.set_ylabel('Sesiones'); ax1.grid(axis='y',alpha=0.3)

# Torta disciplinas
ax2 = axes[1]
dist_disc = df.dropna(subset=['duracion_min']).groupby('disciplina')['duracion_min'].sum()
dist_disc = dist_disc[dist_disc>0].sort_values(ascending=False)
wedges,texts,autotexts = ax2.pie(
    dist_disc.values, labels=dist_disc.index, autopct='%1.0f%%',
    colors=[COLORES.get(d,'gray') for d in dist_disc.index],
    startangle=90, pctdistance=0.75)
for t in autotexts: t.set_fontsize(8)
ax2.set_title('Distribucion del Tiempo por Disciplina', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(REPORTES_DIR,'consistencia_nutricion.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f"\n📊 VALENTINA — Consistencia:")
print(f"   Semanas totales:       {len(ses_semana)}")
print(f"   Excelentes (>=4 ses):  {semanas_excelentes} ({semanas_excelentes/len(ses_semana)*100:.0f}%)")
print(f"   Aceptables (2-3 ses):  {semanas_ok} ({semanas_ok/len(ses_semana)*100:.0f}%)")
print(f"   Pobres (<2 ses):       {semanas_pobres} ({semanas_pobres/len(ses_semana)*100:.0f}%)")
print(f"   Adherencia:            {adherencia:.0f}%")
print(f"\n   {'Consistencia solida — mantener el ritmo!' if adherencia>=60 else 'Mejorar consistencia — meta: 4 sesiones/semana'}")


## �� José — Reporte Ejecutivo Diario + Plan de Entrenamiento
> Síntesis de todos los inputs del equipo. Plan adaptativo basado en HRV.

In [ ]:
# ── REPORTE EJECUTIVO + PLAN SEMANAL ─────────────────────────────
semanas_activas = len(ses_semana[ses_semana>=2])
if semanas_activas<8:    fase = "BASE"
elif semanas_activas<16: fase = "CONSTRUCCION"
else:                     fase = "PICO"

print("="*65)
print(f"REPORTE EJECUTIVO IRONMAN 70.3 — {date.today().strftime('%d %B %Y').upper()}")
print("="*65)
print(f"  Atleta: Lobsang | 42 años | Fase actual: {fase}")
print(f"  Objetivo: Triatlon Sprint Mayo 2026  →  Ironman 70.3")

print(f"\n  ESTADO HOY:")
print(f"    VO2 Max:  {vo2_actual:.1f} ml/kg/min  (Δ +{vo2_actual-vo2_inicio:.1f} desde inicio) ✅")
print(f"    HRV:      {hrv_7d:.1f} ms  →  {semaforo}")
print(f"    RHR:      {rhr_7d:.1f} BPM")
print(f"    Sesiones 14d: {ses_14d}")

print(f"\n  DECISION DEL DIA: {accion.upper()}")

print(f"\n  PLAN SEMANAL TIPO — FASE {fase}:")
plan = [
    ("Lunes",     "Fuerza + Core",            "45 min",   "Isometricos, plancha, deadbug. CACO 4:1 isquio."),
    ("Martes",    "Natacion",                  "50-60 min","Tecnica crol. Bloques progresivos. Meta: 500m continuo."),
    ("Miercoles", "Ciclismo Zona 2 (Azzurri)", "60-75 min","Cadencia 85-95 rpm. HR: 132-143 BPM. Revisar postura."),
    ("Jueves",    "Running CACO",              "40-50 min","Metodo 4:1 → 5:1. Superficie blanda. Pace libre."),
    ("Viernes",   "DESCANSO ACTIVO",           "—",         "Caminata <4 km + movilidad. Sueno objetivo: 7.5h."),
    ("Sabado",    "BRICK (Bici + Run)",        "80-90 min","Bici 20-25 km → transicion → Run 3-5 km. Clave 70.3."),
    ("Domingo",   "DESCANSO TOTAL",            "—",         "Recuperacion. Medir HRV matutino. Planificar semana."),
]
for dia,act,dur,nota in plan:
    print(f"\n    {dia:<12} {act:<32} {dur}")
    print(f"               → {nota}")

print(f"\n  BRECHAS vs IRONMAN 70.3:")
for disc,actual,meta in [('Natacion',km_nado_max,1.9),('Ciclismo',km_bici_max,90),('Running',km_run_max,21.1),('VO2 Max',vo2_actual,45)]:
    pct = actual/meta*100
    bar = '#'*int(pct/5) + '.'*(20-min(20,int(pct/5)))
    print(f"    {disc:<10} [{bar}] {pct:.0f}%  ({actual:.1f} / {meta})")

print(f"\n  ALERTAS:")
print(f"    ⚠️  Isquiotibial izquierdo: protocolo CACO activo")
print(f"    ⚠️  Hombro: monitorear en ciclismo")
print(f"    {'⚠️  HRV bajo — considerar reducir carga' if hrv_7d < hrv_global*0.92 else '✅ HRV OK'}")
print(f"\n  RECOMENDACIONES DEL EQUIPO:")
print(f"    Alejandro: VO2={vo2_actual:.0f} progresa bien. No ignorar señales HRV.")
print(f"    Valentina: Adherencia {adherencia:.0f}%. {'¡Excelente!' if adherencia>=60 else 'Necesitamos mas regularidad.'}")
print(f"    Andrea:    Pre-entreno: avena+banana. Post: 25g proteina + 60g carbos en <45 min.")
print(f"    Hector:    Siguiente meta natacion: 500m continuos sin pull-buoy.")
print(f"    Nairo:     Mantener zona 2. No adelantar volumen antes de resolver isquio.")
print(f"    Catherine: Pace no importa aun — importa el tiempo en zona 2.")

# Guardar reporte texto
txt_path = os.path.join(REPORTES_DIR, f"reporte_{date.today().strftime('%Y%m%d')}.txt")
with open(txt_path,'w',encoding='utf-8') as f:
    f.write(f"REPORTE IRONMAN 70.3 — {date.today()}\n")
    f.write(f"VO2: {vo2_actual:.1f} | HRV 7d: {hrv_7d:.1f} ms ({semaforo}) | RHR: {rhr_7d:.1f}\n")
    f.write(f"Fase: {fase} | Decision: {accion}\n")
    f.write(f"Natacion: {km_nado_max:.2f}/1.9km | Ciclismo: {km_bici_max:.0f}/90km | Running: {km_run_max:.1f}/21.1km\n")
print(f"\n✅ Reporte guardado → reportes/reporte_{date.today().strftime('%Y%m%d')}.txt")
